# Week 10 — Data Mining and Spatial Patterns

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Data Mining · K-means, DBSCAN, clustering

**Learning outcome.** Compare attribute clusters and geographic point clusters.

**What you will implement:** Imputation, scaling, K-means, DBSCAN.  
**Mathematical model:** Silhouette(i)=(b(i)−a(i))/max(a(i),b(i)).

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Silhouette(i)=(b(i)−a(i))/max(a(i),b(i))

**Method family:** Imputation, scaling, K-means, DBSCAN.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
cells=pd.read_csv(DATA/'cells.csv');fac=pd.read_csv(DATA/'facilities.csv')
FEATURES=['canopy_frac','impervious_frac','elevation_m']
Z=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(cells[FEATURES]))
cells['cluster']=KMeans(n_clusters=3,n_init=10,random_state=2026).fit_predict(Z)
print(cells.groupby('cluster')[FEATURES].median().round(3))
plt.scatter(cells.col,cells.row,c=cells.cluster,cmap='Set2',s=110,marker='s');plt.title('Attribute clusters');plt.show()

### Reading the main code

**K-means.** Impute missing attributes, standardize the feature space, and group cells into attribute clusters. The labels are arbitrary.

- `DBSCAN` — Clusters observations by local density under eps and min_samples.
- `StandardScaler` — Centers and scales columns using values learned from training data.
- `SimpleImputer` — Replaces missing predictors with a statistic learned on training data.
- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `KMeans` — Splits feature vectors into a specified number of compact clusters.
- `groupby` — Groups rows before counting or aggregating by sector, date, or other key.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
xy=fac[['easting_m','northing_m']].to_numpy()
fac['cluster']=DBSCAN(eps=430,min_samples=3).fit_predict(xy)
print('Labels:',fac.cluster.value_counts().sort_index())
plt.scatter(fac.easting_m,fac.northing_m,c=fac.cluster,cmap='tab10',s=90);plt.axis('equal')
plt.title('Density clusters in geographic coordinates');plt.show()

### Reading the output code

**DBSCAN.** Density clustering works on projected facility coordinates in metres; `eps` is a distance threshold.

- `DBSCAN` — Clusters observations by local density under eps and min_samples.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Interpretation.** An attribute cluster need not be geographically contiguous. DBSCAN's -1 means noise under the chosen rule, not automatically an error.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Compare K-means attribute clusters with DBSCAN geographic clusters; discuss the role of units.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.